# 🔬 crdt-merge v0.4.0 — A100 Feature Tests

Targeted stress tests for **v0.4.0 features** on NVIDIA A100 GPUs:
- **Provenance tracking** — `merge_with_provenance` at progressive scales
- **Verification at scale** — `verify_crdt` / `verify_convergence` with high trial counts
- **Streaming optimization** — `merge_sorted_stream` with `StreamStats` up to 10M rows
- **System sanity** — quick end-to-end validation of core paths

Expected runtime: **~15–20 minutes** on A100.

Copyright 2026 Ryan Gillespie — Apache-2.0

In [ ]:
# Install dependencies
!pip install -q crdt-merge==0.4.0 psutil

import crdt_merge
print(f"crdt-merge version: {crdt_merge.__version__}")
assert crdt_merge.__version__.startswith("0.4"), f"Expected 0.4.x, got {crdt_merge.__version__}"

import torch, psutil, platform, os
print(f"Python: {platform.python_version()}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"CPUs: {os.cpu_count()}")

In [ ]:
import time, gc, json, tracemalloc, random, string, os
import pandas as pd
import psutil

RESULTS = []
NOTEBOOK_VERSION = "0.4.0"

def record(suite, test, rows=0, duration_s=0.0, throughput=0.0, unit="rows/s",
           mem_mb=0.0, extra=None):
    """Append a result record."""
    entry = {
        "suite": suite,
        "test": test,
        "rows": rows,
        "duration_s": round(duration_s, 4),
        "throughput": round(throughput, 2),
        "unit": unit,
        "mem_mb": round(mem_mb, 2),
        "extra": extra or {},
    }
    RESULTS.append(entry)
    print(f"  ✓ {suite}/{test}: {throughput:,.0f} {unit} | {duration_s:.2f}s | {mem_mb:.1f} MB")

class MemTracker:
    """Context manager for peak memory tracking via tracemalloc."""
    def __enter__(self):
        gc.collect()
        tracemalloc.start()
        self._snap = tracemalloc.take_snapshot()
        return self
    def __exit__(self, *args):
        _, self.peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
    @property
    def peak_mb(self):
        return self.peak / 1e6

def gen_df(n, prefix="A", id_start=0):
    """Generate a DataFrame with n rows."""
    return pd.DataFrame({
        "id": [f"{prefix}_{i}" for i in range(id_start, id_start + n)],
        "value": [random.randint(0, 1_000_000) for _ in range(n)],
        "ts": [float(i) for i in range(n)],
    })

def gen_conflict_pair(n, overlap=0.5):
    """Generate two DataFrames with specified overlap."""
    shared = int(n * overlap)
    only_a = n - shared
    only_b = n - shared
    ids_shared = [f"S_{i}" for i in range(shared)]
    ids_a = [f"A_{i}" for i in range(only_a)]
    ids_b = [f"B_{i}" for i in range(only_b)]
    df_a = pd.DataFrame({
        "id": ids_shared + ids_a,
        "value": [random.randint(0, 1_000_000) for _ in range(shared + only_a)],
        "ts": [float(i) for i in range(shared + only_a)],
    })
    df_b = pd.DataFrame({
        "id": ids_shared + ids_b,
        "value": [random.randint(0, 1_000_000) for _ in range(shared + only_b)],
        "ts": [float(i + 1000) for i in range(shared + only_b)],
    })
    return df_a, df_b

def gen_sorted_stream(n, prefix="S"):
    """Generate a sorted iterable of dicts for streaming tests."""
    for i in range(n):
        yield {"id": f"{prefix}_{i:010d}", "value": random.randint(0, 1_000_000), "ts": float(i)}

def check_ram(required_gb=10):
    avail = psutil.virtual_memory().available / 1e9
    print(f"  Available RAM: {avail:.1f} GB (need {required_gb} GB)")
    return avail >= required_gb

print("Infrastructure ready.")

## Suite 1: PROVENANCE — `merge_with_provenance` at Scale

In [ ]:
from crdt_merge import merge, merge_with_provenance

print("="*70)
print("SUITE 1: PROVENANCE")
print("="*70)

scales = [10_000, 50_000, 100_000, 500_000, 1_000_000]

for n in scales:
    if n >= 500_000 and not check_ram(8):
        print(f"  ⚠ Skipping {n:,} — insufficient RAM")
        continue

    df_a, df_b = gen_conflict_pair(n, overlap=0.5)

    # Plain merge baseline
    gc.collect()
    t0 = time.time()
    plain_result = merge(df_a, df_b, key="id")
    plain_dur = time.time() - t0
    plain_tput = n / plain_dur if plain_dur > 0 else 0

    # Provenance merge
    gc.collect()
    with MemTracker() as mt:
        t0 = time.time()
        prov_result, prov_log = merge_with_provenance(df_a, df_b, key="id", timestamp_col="ts")
        prov_dur = time.time() - t0
    prov_tput = n / prov_dur if prov_dur > 0 else 0

    overhead_pct = ((prov_dur - plain_dur) / plain_dur * 100) if plain_dur > 0 else 0

    record("PROVENANCE", f"merge_with_prov_{n//1000}K",
           rows=n, duration_s=prov_dur, throughput=prov_tput, unit="rows/s",
           mem_mb=mt.peak_mb,
           extra={
               "plain_rows_s": round(plain_tput, 2),
               "overhead_pct": round(overhead_pct, 2),
               "total_conflicts": prov_log.total_conflicts,
               "total_rows": prov_log.total_rows,
               "merged_rows": prov_log.merged_rows,
               "unique_a": prov_log.unique_a_rows,
               "unique_b": prov_log.unique_b_rows,
               "duration_ms": prov_log.duration_ms,
           })

    # Verify provenance log accuracy
    shared = int(n * 0.5)
    assert prov_log.total_conflicts == shared, \
        f"Expected {shared} conflicts, got {prov_log.total_conflicts}"

    # Check summary/to_dict work
    s = prov_log.summary()
    d = prov_log.to_dict()
    assert isinstance(s, str) and len(s) > 0
    assert isinstance(d, dict)

    del df_a, df_b, plain_result, prov_result, prov_log
    gc.collect()

print("\n✅ PROVENANCE suite complete.")

## Suite 2: VERIFY_AT_SCALE — Verification at High Trial Counts

In [ ]:
from crdt_merge import merge, verify_crdt, verify_convergence

print("="*70)
print("SUITE 2: VERIFY_AT_SCALE")
print("="*70)

def gen_fn():
    """Generate a random pair of small DataFrames for verification."""
    n = random.randint(10, 200)
    return gen_conflict_pair(n, overlap=random.uniform(0.2, 0.8))

def merge_fn(a, b):
    return merge(a, b, key="id")

# verify_crdt at increasing trials
for trials in [100, 500, 1000, 2000]:
    gc.collect()
    t0 = time.time()
    result = verify_crdt(merge_fn, gen_fn, trials=trials, include_convergence=True)
    dur = time.time() - t0
    per_trial = (dur / trials * 1000) if trials > 0 else 0

    record("VERIFY_AT_SCALE", f"verify_crdt_{trials}",
           rows=trials, duration_s=dur, throughput=round(trials / dur, 2) if dur > 0 else 0,
           unit="trials/s",
           extra={"ms_per_trial": round(per_trial, 2)})

# verify_convergence at various scales
for trials, replicas in [(100, 5), (500, 5), (100, 10), (500, 10)]:
    gc.collect()
    t0 = time.time()
    result = verify_convergence(merge_fn, gen_fn, trials=trials, num_replicas=replicas)
    dur = time.time() - t0
    per_trial = (dur / trials * 1000) if trials > 0 else 0

    record("VERIFY_AT_SCALE", f"convergence_t{trials}_r{replicas}",
           rows=trials, duration_s=dur, throughput=round(trials / dur, 2) if dur > 0 else 0,
           unit="trials/s",
           extra={"ms_per_trial": round(per_trial, 2), "num_replicas": replicas})

print("\n✅ VERIFY_AT_SCALE suite complete.")

## Suite 3: STREAMING_OPTIMIZATION — `merge_sorted_stream` + `StreamStats`

In [ ]:
from crdt_merge import merge_sorted_stream, StreamStats

print("="*70)
print("SUITE 3: STREAMING_OPTIMIZATION")
print("="*70)

BASELINE_STREAM_ROWS_S = 25_000  # v0.3.0 baseline

scales = [100_000, 500_000, 1_000_000, 5_000_000, 10_000_000]

for n in scales:
    if n >= 5_000_000 and not check_ram(12):
        print(f"  ⚠ Skipping {n:,} — insufficient RAM")
        continue

    stats = StreamStats()
    gc.collect()

    with MemTracker() as mt:
        t0 = time.time()
        # Generate two sorted streams with 50% overlap
        half = n // 2
        source_a = gen_sorted_stream(n, prefix="A")
        source_b = gen_sorted_stream(n, prefix="B")
        row_count = 0
        for batch in merge_sorted_stream(source_a, source_b, key="id",
                                          batch_size=5000, stats=stats):
            row_count += len(batch)
        dur = time.time() - t0

    tput = row_count / dur if dur > 0 else 0
    speedup = tput / BASELINE_STREAM_ROWS_S if BASELINE_STREAM_ROWS_S > 0 else 0

    record("STREAMING", f"stream_{n//1000}K",
           rows=row_count, duration_s=dur, throughput=tput, unit="rows/s",
           mem_mb=mt.peak_mb,
           extra={
               "input_rows": n * 2,
               "batches": stats.batches_processed,
               "peak_batch": stats.peak_batch_size,
               "stats_rows_s": stats.rows_per_sec(),
               "stats_merged": stats.rows_merged,
               "stats_unique_a": stats.rows_unique_a,
               "stats_unique_b": stats.rows_unique_b,
               "speedup_vs_v030": round(speedup, 2),
           })

    # Verify constant memory — should stay under ~15 MB regardless of scale
    if n >= 1_000_000:
        assert mt.peak_mb < 50, \
            f"Memory too high for streaming: {mt.peak_mb:.1f} MB (expected <50 MB)"

    gc.collect()

print("\n✅ STREAMING_OPTIMIZATION suite complete.")

## Suite 4: SYSTEM_SANITY — Quick End-to-End Validation

In [ ]:
from crdt_merge import merge, merge_with_provenance, merge_sorted_stream, StreamStats
from crdt_merge import verify_crdt, verify_convergence

print("="*70)
print("SUITE 4: SYSTEM_SANITY")
print("="*70)

N_SANITY = 500_000

# 4a: Core merge at 500K
print("\n--- 4a: Core merge ---")
df_a, df_b = gen_conflict_pair(N_SANITY, overlap=0.5)
gc.collect()
t0 = time.time()
result = merge(df_a, df_b, key="id")
dur = time.time() - t0
tput = N_SANITY / dur if dur > 0 else 0
record("SANITY", "core_merge_500K", rows=N_SANITY, duration_s=dur, throughput=tput, unit="rows/s")
del result
gc.collect()

# 4b: Strategies at 500K
print("\n--- 4b: Strategies ---")
for strategy in ["lww", "max", "min"]:
    gc.collect()
    t0 = time.time()
    result = merge(df_a, df_b, key="id", strategy=strategy, timestamp_col="ts")
    dur = time.time() - t0
    tput = N_SANITY / dur if dur > 0 else 0
    record("SANITY", f"strategy_{strategy}_500K", rows=N_SANITY, duration_s=dur,
           throughput=tput, unit="rows/s")
    del result
    gc.collect()

del df_a, df_b
gc.collect()

# 4c: Delta compute + apply at 500K
print("\n--- 4c: Delta compute + apply ---")
from crdt_merge import compute_delta, apply_delta
df_old = gen_df(N_SANITY, prefix="OLD")
df_new = gen_df(N_SANITY, prefix="OLD")  # same IDs, different values
df_new["value"] = df_new["value"] + 1

gc.collect()
t0 = time.time()
delta = compute_delta(df_old, df_new, key="id")
dur_compute = time.time() - t0

t0 = time.time()
applied = apply_delta(df_old, delta, key="id")
dur_apply = time.time() - t0

record("SANITY", "delta_compute_500K", rows=N_SANITY, duration_s=dur_compute,
       throughput=N_SANITY / dur_compute if dur_compute > 0 else 0, unit="rows/s")
record("SANITY", "delta_apply_500K", rows=N_SANITY, duration_s=dur_apply,
       throughput=N_SANITY / dur_apply if dur_apply > 0 else 0, unit="rows/s")
del df_old, df_new, delta, applied
gc.collect()

# 4d: CRDT properties at 500 trials
print("\n--- 4d: CRDT properties ---")
t0 = time.time()
vr = verify_crdt(merge_fn, gen_fn, trials=500, include_convergence=True)
dur = time.time() - t0
record("SANITY", "crdt_properties_500", rows=500, duration_s=dur,
       throughput=round(500 / dur, 2) if dur > 0 else 0, unit="trials/s")

# 4e: 5-replica convergence at 100K
print("\n--- 4e: 5-replica convergence ---")

def gen_fn_100k():
    return gen_conflict_pair(100_000, overlap=0.5)

t0 = time.time()
vr = verify_convergence(merge_fn, gen_fn_100k, trials=5, num_replicas=5)
dur = time.time() - t0
record("SANITY", "convergence_5rep_100K", rows=100_000, duration_s=dur,
       throughput=round(5 / dur, 2) if dur > 0 else 0, unit="trials/s")

print("\n✅ SYSTEM_SANITY suite complete.")

## Results Summary

In [ ]:
import json
from datetime import datetime, timezone

print("="*70)
print("RESULTS SUMMARY — crdt-merge v0.4.0 A100 Feature Tests")
print("="*70)

# Group by suite
suites = {}
for r in RESULTS:
    suites.setdefault(r["suite"], []).append(r)

for suite, tests in suites.items():
    print(f"\n{'─'*60}")
    print(f"  {suite}")
    print(f"{'─'*60}")
    for t in tests:
        extra_str = ""
        if t["extra"]:
            highlights = {k: v for k, v in t["extra"].items()
                         if k in ("overhead_pct", "speedup_vs_v030", "ms_per_trial",
                                  "total_conflicts", "num_replicas")}
            if highlights:
                extra_str = " | " + ", ".join(f"{k}={v}" for k, v in highlights.items())
        print(f"  {t['test']:40s} {t['throughput']:>12,.0f} {t['unit']:10s} "
              f"{t['duration_s']:>8.2f}s  {t['mem_mb']:>7.1f} MB{extra_str}")

# Export JSON
report = {
    "notebook": "crdt-merge v0.4.0 A100 Feature Tests",
    "version": NOTEBOOK_VERSION,
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "results": RESULTS,
    "total_tests": len(RESULTS),
    "suites": list(suites.keys()),
}

json_path = "crdt_merge_v040_a100_results.json"
with open(json_path, "w") as f:
    json.dump(report, f, indent=2)
print(f"\n📁 Results saved to {json_path}")

# Try Google Drive save
try:
    from google.colab import drive
    drive.mount("/content/drive")
    drive_path = "/content/drive/MyDrive/crdt-merge-benchmarks/"
    os.makedirs(drive_path, exist_ok=True)
    import shutil
    shutil.copy(json_path, drive_path + json_path)
    print(f"📁 Also saved to Google Drive: {drive_path}{json_path}")
except Exception as e:
    print(f"ℹ️  Google Drive save skipped: {e}")

print(f"\n🏁 All {len(RESULTS)} tests complete.")